In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW
import math

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- TOKENIZER --------
tok = GPT2Tokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token

# -------- DATA --------
class DS(Dataset):
    def __init__(self, texts):
        self.enc = tok(texts, padding="max_length",
                       truncation=True, max_length=128)

    def __getitem__(self,i):
        item = {k:torch.tensor(v[i]) for k,v in self.enc.items()}
        item["labels"] = item["input_ids"].clone()
        return item

    def __len__(self):
        return len(self.enc["input_ids"])

texts = [
    "User: Hello\nBot: Hi!",
    "User: Tell me a joke\nBot: Why did the cat laugh?"
]

loader = DataLoader(DS(texts), batch_size=2)

# -------- MODEL --------
model = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
opt = AdamW(model.parameters(), lr=5e-5)

# -------- TRAIN --------
for e in range(3):
    model.train()
    total = 0
    for b in loader:
        b = {k:v.to(DEVICE) for k,v in b.items()}
        out = model(**b)
        loss = out.loss

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item()

    print(f"Epoch {e+1} Loss={total/len(loader):.4f}")

# -------- PERPLEXITY --------
def ppl(loss):
    return math.exp(loss)

print("Example Perplexity:", ppl(1.5))

# -------- GENERATION --------
def gen(text,temp):
    inp = tok(text, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inp,
        max_length=100,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=temp
    )
    return tok.decode(out[0], skip_special_tokens=True)

for t in [0.5,0.7,1.0,1.2]:
    print("\nTemp:",t)
    print(gen("User: Hello\nBot:",t))

# -------- ZERO-SHOT COMPARISON --------
base = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)

def zero_shot(text):
    inp = tok(text, return_tensors="pt").to(DEVICE)
    out = base.generate(**inp, max_length=100)
    return tok.decode(out[0], skip_special_tokens=True)

print("\nZero-shot:")
print(zero_shot("User: Hello\nBot:"))